# Decision Tree ID3 Algorithm Demonstration

This notebook demonstrates the ID3 algorithm for building a decision tree using Spotify dataset. We'll classify tracks as explicit or not based on various features.

In [16]:
import pandas as pd
import numpy as np
from collections import Counter
import math

In [ ]:
# Use the dataset from the image (buys computer)
data = [
    {'Age': '<=30', 'Income': 'High', 'Student': 'No', 'Credit Rating': 'Fair', 'Buys Computer': 'No'},
    {'Age': '<=30', 'Income': 'High', 'Student': 'No', 'Credit Rating': 'Excellent', 'Buys Computer': 'No'},
    {'Age': '31-40', 'Income': 'High', 'Student': 'No', 'Credit Rating': 'Fair', 'Buys Computer': 'Yes'},
    {'Age': '>40', 'Income': 'Medium', 'Student': 'No', 'Credit Rating': 'Fair', 'Buys Computer': 'Yes'},
    {'Age': '>40', 'Income': 'Low', 'Student': 'Yes', 'Credit Rating': 'Fair', 'Buys Computer': 'Yes'},
    {'Age': '>40', 'Income': 'Low', 'Student': 'Yes', 'Credit Rating': 'Excellent', 'Buys Computer': 'No'},
    {'Age': '31-40', 'Income': 'Low', 'Student': 'Yes', 'Credit Rating': 'Excellent', 'Buys Computer': 'Yes'},
    {'Age': '<=30', 'Income': 'Medium', 'Student': 'No', 'Credit Rating': 'Fair', 'Buys Computer': 'No'},
    {'Age': '<=30', 'Income': 'Low', 'Student': 'Yes', 'Credit Rating': 'Fair', 'Buys Computer': 'Yes'},
    {'Age': '>40', 'Income': 'Medium', 'Student': 'Yes', 'Credit Rating': 'Fair', 'Buys Computer': 'Yes'},
    {'Age': '<=30', 'Income': 'Medium', 'Student': 'Yes', 'Credit Rating': 'Excellent', 'Buys Computer': 'Yes'},
    {'Age': '31-40', 'Income': 'Medium', 'Student': 'No', 'Credit Rating': 'Excellent', 'Buys Computer': 'Yes'},
    {'Age': '31-40', 'Income': 'High', 'Student': 'Yes', 'Credit Rating': 'Fair', 'Buys Computer': 'Yes'},
    {'Age': '>40', 'Income': 'Medium', 'Student': 'No', 'Credit Rating': 'Excellent', 'Buys Computer': 'No'},
]

df = pd.DataFrame(data)

# Select relevant features and target
features = ['Age', 'Income', 'Student', 'Credit Rating']
target = 'Buys Computer'

# No missing values expected in this dataset
df_clean = df[features + [target]].copy()

print("Dataset shape:", df_clean.shape)
print("Features:", features)
print("Target:", target)
print("\nSample data:")
print(df_clean.head())

Dataset shape: (14, 5)
Features: ['Age', 'Income', 'Student', 'Credit Rating']
Target: Buys Computer

Sample data:
     Age  Income Student Credit Rating Buys Computer
0   <=30    High      No          Fair            No
1   <=30    High      No     Excellent            No
2  31-40    High      No          Fair           Yes
3    >40  Medium      No          Fair           Yes
4    >40     Low     Yes          Fair           Yes


In [18]:
class Node:
    def __init__(self, feature=None, label=None):
        self.feature = feature      # feature to split on
        self.label = label          # class label if leaf
        self.children = {}          # feature_value -> Node


import math
from collections import Counter

def entropy(labels):
    total = len(labels)
    counts = Counter(labels)

    e = 0
    for c in counts.values():
        p = c / total
        e -= p * math.log2(p)

    return e


def information_gain(data, feature, target):
    base_entropy = entropy(data[target])
    total = len(data)

    gain = base_entropy

    for value in data[feature].unique():
        subset = data[data[feature] == value]
        weight = len(subset) / total
        gain -= weight * entropy(subset[target])

    return gain


def id3(data, features, target, depth=0, max_depth=5):

    # If all labels same → leaf
    if len(data[target].unique()) == 1:
        return Node(label=data[target].iloc[0])

    # Stop if no features or depth limit
    if not features or depth == max_depth:
        return Node(label=data[target].mode()[0])

    # Choose best feature
    best = max(features, key=lambda f: information_gain(data, f, target))
    root = Node(feature=best)

    remaining = [f for f in features if f != best]

    # Split
    for value in data[best].unique():
        subset = data[data[best] == value]

        if subset.empty:
            root.children[value] = Node(label=data[target].mode()[0])
        else:
            root.children[value] = id3(
                subset, remaining, target, depth + 1, max_depth
            )

    return root


def print_tree(node, indent=""):
    if node.label is not None:
        print(indent + "Leaf →", node.label)
        return

    print(indent + "Feature →", node.feature)

    for value, child in node.children.items():
        print(indent + f"  [{value}]")
        print_tree(child, indent + "    ")


def classify(row, node):
    if node.label is not None:
        return node.label

    value = row[node.feature]

    if value in node.children:
        return classify(row, node.children[value])

    return None  # unseen value


In [19]:
# Build the decision tree
tree = id3(df_clean, features, target, max_depth=3)

# Print the tree
print("\nDecision Tree Structure:")
print_tree(tree)

# Test classification on samples
print("\nSample Predictions:")
samples = df_clean.sample(5, random_state=42)

correct = 0
total = len(df_clean)

for _, row in df_clean.iterrows():
    pred = classify(row, tree)
    if pred is not None and pred == row[target]:
        correct += 1

accuracy = correct / total * 100
print(f"\nTraining Accuracy: {accuracy:.2f}%")



Decision Tree Structure:
Feature → Age
  [<=30]
    Feature → Student
      [No]
        Leaf → No
      [Yes]
        Leaf → Yes
  [31-40]
    Leaf → Yes
  [>40]
    Feature → Credit Rating
      [Fair]
        Leaf → Yes
      [Excellent]
        Leaf → No

Sample Predictions:

Training Accuracy: 100.00%


In [20]:
# Classify a new sample
print("Classifying a new sample...")

# Create a new sample (hypothetical person)
new_sample = {
    'Age': '<=30',
    'Income': 'Medium',
    'Student': 'Yes',
    'Credit Rating': 'Fair'
}

prediction = classify(pd.Series(new_sample), tree)
print(f"New sample features: {new_sample}")
print(f"Predicted class (Buys Computer): {prediction}")
print("No = Does not buy, Yes = Buys")

# Another sample
new_sample2 = {
    'Age': '>40',
    'Income': 'High',
    'Student': 'No',
    'Credit Rating': 'Excellent'
}

prediction2 = classify(pd.Series(new_sample2), tree)
print(f"\nSecond sample features: {new_sample2}")
print(f"Predicted class (Buys Computer): {prediction2}")

Classifying a new sample...
New sample features: {'Age': '<=30', 'Income': 'Medium', 'Student': 'Yes', 'Credit Rating': 'Fair'}
Predicted class (Buys Computer): Yes
No = Does not buy, Yes = Buys

Second sample features: {'Age': '>40', 'Income': 'High', 'Student': 'No', 'Credit Rating': 'Excellent'}
Predicted class (Buys Computer): No
